
# Title: HRS Health Silver CDM DDL Functional ETL Specification
________________________________________
## 1.	Document Information
- Property	Value
- Document Name	HRS Silver CDM DDL Functional Specification
- Version	1.0
- Author	Perez
- Last Updated	2026-07-20
- Target Runtime	Databricks Runtime 15.x
- SQL Dialect	Spark SQL
- Storage Format	Delta Lake
- Deployment Environment	Development

________________________________________
## 2.	Objective
Generate a Databricks SQL load script using Spark SQL compatible with Databricks Runtime 15.x.
The generated SQL will load one HRS survey section from the RAND HRS source table into the corresponding Silver CDM table.
Return only SQL.
________________________________________
## 3.	Purpose
This specification defines the business rules, source-to-target mappings, lookup rules, and SQL generation requirements for loading a single HRS survey section.
The generated SQL must conform to the Silver Common Data Model (CDM).
________________________________________
## 4.	Parameters
These are the only values that should change between survey sections.  There are two parts separated by a comma.  The first part is the parameter name and the second part is the value.
- Parameter, 	Value
- SECTION_NAME,	health
- SECTION_DESCRIPTION,	RAND HRS Codebook – health
- SOURCE_TABLE,	dev_catalog.brz_raw_hrs.randhrs1992_2022v1
- TARGET_TABLE,	dev_catalog.slv_cdm_hrs.hrs_health
- RESPONDENT_TABLE,	dev_catalog.slv_cdm_hrs.hrs_survey_respondent
- PRIMARY_KEY,	hrs_health_id
- FOREIGN_KEY,	hrs_survey_respondent_id
________________________________________
## 5.	Source Objects
### Source Table
- dev_catalog.brz_raw_hrs.randhrs1992_2022v1

### Reference Tables
- dev_catalog.slv_cdm_hrs.hrs_survey_respondent
________________________________________
## 6.	Target Data Model
### Parent Table
- RESPONDENT_TABLE 
### Child Table
- TARGET_TABLE
### Relationship
- One RESPONDENT_TABLE record to  Many  TARGET_TABLE records
### Business Rules
- Each TARGET_TABLE record belongs to exactly one RESPONDENT_TABLE record. 
- Every TARGET_TABLE record must contain a valid FOREIGN_KEY. 
- A RESPONDENT_TABLE record may have one record for each survey section. 
________________________________________
## 7.	Keys
### Primary Key
- PRIMARY_KEY 
    - Identity column 
    - Auto-generated 
    - Do not populate 
Foreign Key
o	FOREIGN_KEY 
o	Populate using the Respondent lookup.
________________________________________
## 8.	Dependencies
- The following objects must already exist before execution.
-   RESPONDENT_TABLE
- The RESPONDENT_TABLE must be populated prior to loading the TARGET_TABLE.
## 9.	Load Strategy
- INSERT INTO
    - Generate one target record for every qualifying respondent. 
    - Do not update existing records. 
    - Do not truncate the target table. 
    - No UPDATE
    - No DELETE
    - No Merge
    - Append only
## 10.	Restart Behavior
- The script must be restartable. 
- Rerunning the load should not create duplicate rows.
________________________________________
## 11.	Business Rules
- Generate exactly one target record for every respondent contained in the source table.
- Each target record contains every attribute belonging to the survey section.
- Copy all mapped source columns directly.
- Do not
    - Aggregate 
    - Pivot 
    - Unpivot 
    - Summarize 
    - Modify values 
- Preserve
    - NULL values 
    - Original data types (unless explicitly stated) 
    - Original source values 
________________________________________
## 12.	Null Handling
- Preserve NULL values.
- Unless explicitly instructed, Do not, 
    - Replace NULL 
    - Default NULL 
    - Use COALESCE() 
________________________________________
## 13.	Source-to-Target Mapping
There are three parts separated by a comma in the table below.  The first part is the Target Column, then the Source Column, then the Rule.

    Target Column,	Source Column,	Rule

- HHIDPN,	HHIDPN,	Direct Copy
- R1BMI,	R1BMI,	Direct Copy
- ...,	...,	Direct Copy
- R16BMI,	R16BMI,	Direct Copy

Generate the INSERT statement using the target column order shown above.
________________________________________
## 14.	Lookup Rules
Respondent Lookup
- Join source.HHIDPN = hrs_survey_respondent.HHIDPN.
- Populate FOREIGN_KEY from the lookup table.
- Use INNER JOIN
- Exclude rows where no respondent exists.
________________________________________
## 15.	Audit Columns
Populate the following target columns.  There are wo parts separated by a comma in the table below.  The first part is the Target Column, then the value.

    Target Column,	Value
- create_date,	CURRENT_DATE()
- update_date,	CURRENT_DATE()
- active,	TRUE
________________________________________
## 16.	Duplicate Handling
- A health record is uniquely identified by HHIDPN.
- Before inserting a record, verify that HHIDPN does not already exist in TARGET_TABLE. 
- Skip duplicate records.
 ________________________________________
## 17.	Performance Requirements
- Use a single INSERT statement. 
- Avoid repeated scans of SOURCE_TABLE. 
- Avoid unnecessary subqueries. 
- Generate readable SQL. 
- Optimize joins where practical.
## 18.	Error Handling
Exclude records when
- respondent lookup fails 
- Data type conversion fails 
- Required source columns missing 
- Unexpected NULL primary identifiers
- required foreign keys cannot be resolved 

Do not
- create placeholder values 
- insert partial records 
- invent reference keys 
________________________________________
## 19.	Naming Standards
- Use aliases: src, resp, tgt, 
- Use descriptive CTE names when needed.
________________________________________
## 20.	SQL Requirements
Generate
- one INSERT INTO statement 
- explicit target column list 
- explicit SELECT column list 

Do not use SELECT *

Use
- meaningful aliases 
- INNER JOIN 
- uppercase SQL keywords 
- consistent indentation

Return only SQL.
________________________________________
## 21.	Out of Scope
Do not 
- Create tables 
- Drop tables 
- Alter schema 
- Modify indexes 
- Modify constraints 
- Generate DDL 
- Generate test scripts 
- Generate validation scripts

Only generate the requested DML.
________________________________________
## 22.	Add Comments Requirements
- Comment each major SQL section. 
- Include comments for Lookup logic 
- Business rules 
- Audit column population
________________________________________
## 23.	Assumptions
Assume
- lookup tables are populated 
- source schema is correct 
- target schema already exists 

Use only
- tables 
- columns 
- relationships defined in this specification.

Do not invent
- tables 
- columns 
- joins 
- transformations 
________________________________________
## 24.	Validation Requirements
- The generated SQL should satisfy the following:
    - One target row per respondent
    - No duplicate HHIDPN values
    - No orphaned records

- Every target row references a valid respondent
- Audit columns populated
- Primary key omitted (identity column)
- Identity column not populated
- Direct column mapping preserved
- No unnecessary transformations
________________________________________
## 25.	Deliverable
Generate a single Databricks SQL script named: /sql/dml/load_<SECTION_NAME>.sql

Return only SQL.
